# Ekstraksi Fitur RMS dari Audio MP3 (Berisik vs Non-Berisik)

Notebook ini dijalankan **lokal di VS Code**. Letakkan file `.ipynb` ini sejajar dengan folder `berisik` dan `non-berisik`:

```
folder_proyek/
├── ekstraksi_fitur_rms_vscode.ipynb
├── berisik/
│   ├── xxx.mp3
│   └── ...
└── non-berisik/
    ├── yyy.mp3
    └── ...
```

Hasilnya satu file `fitur_rms.csv` berisi semua audio dengan kolom label.

## 1. Instalasi Library

In [ ]:
%pip install soundfile pandas numpy openpyxl joblib tqdm ipywidgets -q

Note: you may need to restart the kernel to use updated packages.


: 

## 2. Fitur RMS (Root Mean Square)

### Penjelasan Fitur
RMS adalah ukuran **energi / kekuatan rata-rata sinyal audio**. Semakin besar nilai RMS, semakin keras (besar amplitudonya) suara tersebut. RMS sering dipakai untuk mengukur loudness, mendeteksi bagian hening, dan sebagai fitur klasifikasi audio. Pada kasus ini, RMS dipakai untuk membedakan audio **berisik** dan **non-berisik**.

### Rumus
RMS keseluruhan sinyal:

$$RMS = \sqrt{\frac{1}{N}\sum_{n=1}^{N} x[n]^2}$$

RMS per frame:

$$RMS_t = \sqrt{\frac{1}{K}\sum_{k=1}^{K} x_t[k]^2}$$

### Penjelasan Rumus
- $x[n]$ : nilai amplitudo sinyal audio pada sampel ke-n
- $N$ : jumlah total sampel pada audio
- $x_t[k]$ : sampel ke-k pada frame ke-t
- $K$ : jumlah sampel dalam satu frame (`frame_length`)

Langkah perhitungan:
1. **Square (kuadrat)**: setiap sampel dikuadratkan agar nilai negatif menjadi positif.
2. **Mean (rata-rata)**: semua hasil kuadrat dijumlahkan lalu dibagi jumlah sampel.
3. **Root (akar)**: hasil rata-rata diakarkan agar satuannya kembali sama seperti amplitudo asli.

Karena audio dipotong menjadi banyak frame, hasilnya berupa deret nilai RMS. Untuk mendapat satu fitur per file, nilainya diringkas dengan **mean, std, max, dan min**.

## 3. Ekstraksi RMS dari Semua Audio

Folder dataset dibaca dari lokasi notebook ini (`folder_dataset = "."`). Kalau nama folder kamu beda, ubah di bagian **PENGATURAN**.

In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
from numpy.lib.stride_tricks import sliding_window_view
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ====== PENGATURAN ======
folder_dataset = "audio"                 # "." = folder tempat notebook ini berada
frame_length = 2048
hop_length = 512
# nama folder -> label angka (0 = non-berisik, 1 = berisik)
label_map = {"non-berisik": 0, "berisik": 1}
# ========================

print("Folder kerja:", os.path.abspath(folder_dataset))

def cari_mp3(folder):
    hasil = []
    for root, _, fnames in os.walk(folder):
        for f in fnames:
            if f.lower().endswith(".mp3"):     # .lower() supaya .MP3 juga ikut terbaca
                hasil.append(os.path.join(root, f))
    return sorted(hasil)

def baca_audio(path):
    try:
        y, sr = sf.read(path, dtype="float32", always_2d=True)
        return y.mean(axis=1), sr               # jadikan mono
    except Exception:
        import librosa                          # cadangan (butuh: pip install librosa + ffmpeg)
        return librosa.load(path, sr=None, mono=True)

def rms_per_frame(y):
    # setara librosa.feature.rms (center=True, padding nol)
    y = np.pad(y, frame_length // 2)
    frames = sliding_window_view(y, frame_length)[::hop_length]
    return np.sqrt(np.mean(frames.astype(np.float64) ** 2, axis=1))

def proses(path, label):
    try:
        y, sr = baca_audio(path)
        rms = rms_per_frame(y)
        return {
            "label": label,
            "label_num": label_map[label],
            "filename": os.path.basename(path),
            "sr": sr,
            "durasi_detik": round(len(y) / sr, 2),
            "rms_global": float(np.sqrt(np.mean(y.astype(np.float64) ** 2))),
            "rms_mean": float(rms.mean()),
            "rms_std": float(rms.std()),
            "rms_max": float(rms.max()),
            "rms_min": float(rms.min()),
        }
    except Exception as e:
        print(f"Gagal memproses {path}: {e}")
        return None

# Kumpulkan semua file dari kedua folder
tugas = []
for label in label_map:
    daftar = cari_mp3(os.path.join(folder_dataset, label))
    print(f"{label}: ditemukan {len(daftar)} file")
    tugas += [(p, label) for p in daftar]

if not tugas:
    raise FileNotFoundError("Tidak ada file mp3. Cek posisi notebook dan nama folder di PENGATURAN.")

# Proses paralel + progress bar
hasil = Parallel(n_jobs=-1)(
    delayed(proses)(p, l) for p, l in tqdm(tugas, desc="Ekstraksi RMS")
)

# Gabung jadi satu tabel (satu CSV, ada kolom label)
df = pd.DataFrame([h for h in hasil if h is not None])
print("Total file diproses:", len(df))
df

## 4. Ringkasan per Kelas

In [ ]:
kolom_fitur = ["rms_global", "rms_mean", "rms_std", "rms_max", "rms_min"]
df.groupby("label")[kolom_fitur].agg(["mean", "min", "max"]).round(4).T

## 5. Simpan Hasil (Satu File Gabungan)

File disimpan di folder yang sama dengan notebook. Kolom `label` berisi teks (`berisik` / `non-berisik`) dan `label_num` berisi angka (`1` / `0`) untuk keperluan model.

In [ ]:
df.to_csv("fitur_rms.csv", index=False)
df.to_excel("fitur_rms.xlsx", index=False)
print("Tersimpan:", os.path.abspath("fitur_rms.csv"))